# 04 - Gemini com LangChain

O `ChatGoogleGenerativeAI` integra o Gemini ao ecossistema LangChain.
A interface é idêntica ao `ChatOpenAI` e `ChatGroq` — trocar de provedor é só mudar o import.

### Pacotes necessários

```bash
pip install langchain-google-genai python-dotenv
```

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(find_dotenv())

api_key = os.environ.get("GEMINI_API_KEY")

chat = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # atualizado do gemini-1.5-flash
    temperature=0,
    api_key=api_key
)

## 1. Chat básico com mensagens de sistema

O `ChatGoogleGenerativeAI` aceita o formato de mensagens do LangChain diretamente —
tuplas `(role, content)` com `system`, `human` e `ai`.

In [2]:
mensagens = [
    ("system", "Você é um assistente que atua como tradutor de português para espanhol. Traduza a seguinte frase"),
    ("human", "Aprender sobre automação, análise de dados e inteligência artificial é importante")
]

response = chat.invoke(mensagens)
print(response.content)

Aprender sobre automatización, análisis de datos e inteligencia artificial es importante.


## 2. Streaming com LCEL

O `.stream()` funciona nativamente com o `ChatGoogleGenerativeAI`.

In [3]:
mensagens_stream = [
    ("system", "Você é um assistente que fala muito"),
    ("human", "Olá")
]

for chunk in chat.stream(mensagens_stream):
    print(chunk.content, end="", flush=True)

Olá, olá, olá! Que maravilha ter você por aqui! Já estava aqui pensando em mil e uma coisas para conversar, e de repente, *puf*, você aparece! Que bom!

Me diga, o que te traz por essas bandas hoje? Ou melhor, como posso iluminar o seu dia com uma boa conversa? Estou aqui, prontíssimo para tagarelar sobre qualquer coisa que você queira! O que está na sua mente? O que te deixou curioso? Ou talvez só queira bater um papo descontraído? Conte-me tudo! Estou com os ouvidos e a "boca" (metaforicamente falando, claro!) abertos! 😊

## 3. Chain com ChatPromptTemplate

O Gemini funciona perfeitamente com o operador `|` do LCEL — mesma sintaxe de qualquer outro provedor.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = ChatPromptTemplate.from_messages([
    ("system", "Você é um especialista em {area}. Responda de forma clara e didática."),
    ("human", "{pergunta}")
])

chain = template | chat | StrOutputParser()

response = chain.invoke({
    "area": "inteligência artificial",
    "pergunta": "O que é transfer learning?"
})
print(response)

Olá! Como especialista em Inteligência Artificial, vou explicar o Transfer Learning de uma forma bem clara e didática.

---

### O que é Transfer Learning (Aprendizado por Transferência)?

Imagine que você já aprendeu a andar de bicicleta. Agora, alguém te pede para aprender a andar de moto. Você não precisa começar do zero, aprendendo novamente sobre equilíbrio, direção e como usar os freios. Você já tem uma base sólida de conhecimento (andar de bicicleta) que pode ser *transferida* e adaptada para a nova tarefa (andar de moto), focando apenas nas diferenças (acelerador, embreagem, etc.).

**Transfer Learning é exatamente isso no mundo da Inteligência Artificial:** é uma técnica onde um modelo de IA, que já foi treinado para uma tarefa específica, é reutilizado como ponto de partida para uma nova tarefa, que é *relacionada* à original.

Em vez de construir e treinar um modelo do zero para cada nova tarefa, nós "pegamos emprestado" o conhecimento que um modelo já adquiriu e o adaptamos

## Resumo

| Operação | Código |
|----------|--------|
| Criar modelo | `ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)` |
| Chat direto | `chat.invoke([("system", ...), ("human", ...)])` |
| Streaming | `chat.stream(mensagens)` |
| Chain LCEL | `template \| chat \| StrOutputParser()` |

> **Vantagem do Gemini**: janela de contexto de até 2M tokens no `gemini-1.5-pro` —
> ideal para processar documentos muito longos que outros modelos não conseguem.